# Block 4: Retrieval Evaluation — Recall@K, nDCG@10, Spearman ρ

This notebook covers the quantitative retrieval evaluation block (v2 output — 6 rankers × 3 reference signals).

## Sections Overview
1. **Load Results** — Load and display overall metrics
2. **Per-Reference Signal Comparison** — Compare rankers across Similarity, LiquidityUplift, and Utility references
3. **Component Breakdown Analysis** — Analyze what drives top-10 results (return/sector/liquidity composition)
4. **Per-Query Drill-Down** — Detailed analysis for individual queries
5. **Overall Metrics** — Performance summary by sector, market cap, liquidity tier
6. **Metric Deep-Dives** — nDCG vs Recall analysis
7. **Per-Reference Signal Comparison (Detailed)** — Scatter plots by reference
8. **Utility Breakdown** — Individual query breakdown tables
9. **Radar Chart** — Multi-dimensional ranker comparison
10. **Re-run Retrieval Evaluation** — Execute pipeline from notebook
11. **Summary & Interpretation** — Key findings and recommendations

| Metric | What it measures |
|--------|------------------|
| **Recall@10** | Fraction of ground-truth peers found in top-10 |
| **nDCG@10** | Ranked quality — rewards placing most-relevant peers at top |
| **Spearman ρ** | Rank correlation between predicted ranking and reference ordering |

**Rankers compared:**
- `embedding` — dual-encoder cosine similarity (baseline)
- `pearson_corr` — 60-day Pearson return correlation
- `spearman_corr` — 60-day Spearman return correlation
- `embedding_rerank` — embedding re-ranked by liquidity proximity
- `pearson_corr_rerank` — Pearson correlation re-ranked by liquidity proximity
- `spearman_corr_rerank` — Spearman correlation re-ranked by liquidity proximity

**Reference signals (ground truth):**
- **Similarity** — composite: return correlation + sector + size similarity
- **LiquidityUplift** — did the candidate improve the query's liquidity access?
- **Utility** — blended utility score (return quality × liquidity improvement)


In [ ]:
import sys
from pathlib import Path

# Resolve project root whether Jupyter is launched from project root
# (standard: `uv run jupyter notebook`) or from the notebooks/ sub-dir.
_cwd = Path().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

METRICS_DIR = PROJECT_ROOT / "results" / "retrieval_v2" / "metrics"

# Ranker display names and palette (keys match CSV column names exactly)
RANKER_LABELS = {
    "embedding"           : "Embedding",
    "pearson_corr"        : "Pearson Corr.",
    "spearman_corr"       : "Spearman Corr.",
    "embedding_rerank"    : "Emb. + Rerank",
    "pearson_corr_rerank" : "Pearson + Rerank",
    "spearman_corr_rerank": "Spearman + Rerank",
}
RANKER_COLORS = [
    "#2E86AB", "#A23B72", "#F18F01", "#06A77D", "#C73E1D", "#7B2D8B",
]

# Reference signal labels
REF_LABELS = {
    "similarity"      : "Similarity",
    "liquidity_uplift": "LiquidityUplift",
    "utility"         : "Utility",
}

print(f"Project root   : {PROJECT_ROOT}")
print(f"Metrics dir    : {METRICS_DIR}")

---
## 1. Load Results

In [ ]:
# Overall metrics (averaged across all 3 reference signals)
overall_df = pd.read_csv(METRICS_DIR / "retrieval_metrics_overall.csv")

# Per-reference breakdown
sim_df   = pd.read_csv(METRICS_DIR / "retrieval_similarity.csv")
liq_df   = pd.read_csv(METRICS_DIR / "retrieval_liquidity_uplift.csv")
util_df  = pd.read_csv(METRICS_DIR / "retrieval_utility.csv")

# Utility quality breakdown (% return-sim, % sector-match, % liquidity-improve)
util_bk  = pd.read_csv(METRICS_DIR / "utility_breakdown_averaged.csv")

# Per-query detail (for distribution analysis)
detail_path = METRICS_DIR / "retrieval_metrics_6x3_detailed.csv"
detail_df = pd.read_csv(detail_path) if detail_path.exists() else pd.DataFrame()

# Reference DataFrames keyed by short name
ref_dfs = {
    "similarity"      : sim_df,
    "liquidity_uplift": liq_df,
    "utility"         : util_df,
}

# Derive canonical ranker list from actual CSV columns (preserves order, drops missing)
rankers = [r for r in RANKER_LABELS if r in overall_df.columns]

print(f"Rankers  : {rankers}")
print(f"Metrics  : {overall_df['metric_name'].tolist()}")
print("\nOverall metrics (avg across 3 references):")
overall_df


---
## 2. Per-Reference Signal Comparison

Compare how each ranker performs against different ground truth references:
- **Similarity** — composite: return correlation + sector + size
- **LiquidityUplift** — did the candidate improve liquidity access?
- **Utility** — blended utility score (similarity × liquidity improvement)

In [ ]:
# Load per-reference metrics
similarity_df = pd.read_csv(METRICS_DIR / "retrieval_similarity.csv")
liquidity_df = pd.read_csv(METRICS_DIR / "retrieval_liquidity_uplift.csv")
utility_df = pd.read_csv(METRICS_DIR / "retrieval_utility.csv")

print("Similarity reference metrics:")
print(similarity_df.to_string(index=False))
print("\nLiquidityUplift reference metrics:")
print(liquidity_df.to_string(index=False))
print("\nUtility reference metrics:")
print(utility_df.to_string(index=False))

In [ ]:
# Create side-by-side comparison heatmapfig, axes = plt.subplots(1, 3, figsize=(18, 5))refs = {    "Similarity": similarity_df,    "LiquidityUplift": liquidity_df,    "Utility": utility_df,}for ax, (ref_name, df) in zip(axes, refs.items()):    # Prepare data for heatmap    heatmap_data = df.set_index("metric_name")[list(RANKER_LABELS.keys())].rename(columns=RANKER_LABELS)        sns.heatmap(        heatmap_data,        annot=True,        fmt=".3f",        cmap="RdYlGn",        vmin=0,        vmax=1,        cbar_kws={"label": "Score"},        ax=ax,    )    ax.set_title(f"{ref_name} Reference", fontweight="bold")    ax.set_xlabel("Ranker")    ax.set_ylabel("Metric")plt.suptitle("Retrieval Metrics by Reference Signal", fontsize=14, fontweight="bold", y=1.02)plt.tight_layout()(METRICS_DIR / "../figures").mkdir(parents=True, exist_ok=True)plt.savefig(METRICS_DIR / "../figures/metrics_by_reference_heatmap.png", dpi=150, bbox_inches="tight")plt.show()

---
## 3. Component Breakdown Analysis

Analyze what drives each ranker's top-10 results:
- **Return similarity**: Behavioral similarity (60-day return correlation)
- **Sector similarity**: Industry/sector match
- **Liquidity improvement**: Positive LiquidityUplift (more liquid than query)

In [ ]:
# Load utility breakdown analysis
breakdown_path = METRICS_DIR / "utility_breakdown_averaged.csv"

if breakdown_path.exists():
    breakdown_df = pd.read_csv(breakdown_path)
    print("Utility Breakdown (averaged across all queries):")
    print(breakdown_df.to_string())
else:
    print(f"Warning: {breakdown_path} not found. Run evaluation first.")
    breakdown_df = None

In [ ]:
if breakdown_df is not None:    # Prepare data    breakdown_df = breakdown_df.reset_index()    breakdown_df = breakdown_df.rename(columns={        "pct_return_sim": "Return Similarity (%)",        "pct_sector_sim": "Sector Similarity (%)",        "pct_liq_improve": "Liquidity Improvement (%)",    })        # Rename rankers for display    breakdown_df["ranker"] = breakdown_df["ranker"].map(RANKER_LABELS)        # Create stacked bar chart    fig, ax = plt.subplots(figsize=(12, 6))        x = np.arange(len(breakdown_df))    width = 0.6        p1 = ax.bar(x, breakdown_df["Return Similarity (%)"], width, label="Return Similarity", color="#2E86AB")    p2 = ax.bar(x, breakdown_df["Sector Similarity (%)"], width,                 bottom=breakdown_df["Return Similarity (%)"],                 label="Sector Similarity", color="#A23B72")    p3 = ax.bar(x, breakdown_df["Liquidity Improvement (%)"], width,                bottom=breakdown_df["Return Similarity (%)"] + breakdown_df["Sector Similarity (%)"],                label="Liquidity Improvement", color="#06A77D")        ax.set_xlabel("Ranker", fontweight="bold")    ax.set_ylabel("Percentage of Top-10 Results (%)", fontweight="bold")    ax.set_title("What Drives Each Ranker's Top-10 Results?\n(Component Breakdown)",                  fontsize=14, fontweight="bold")    ax.set_xticks(x)    ax.set_xticklabels(breakdown_df["ranker"], rotation=45, ha="right")    ax.legend(loc="upper left", bbox_to_anchor=(1, 1))    ax.set_ylim(0, 105)        # Add value labels    for i, row in breakdown_df.iterrows():        total = row["Return Similarity (%)"] + row["Sector Similarity (%)"] + row["Liquidity Improvement (%)"]        ax.annotate(f"{total:.1f}%", (i, total + 2), ha="center", va="bottom", fontsize=9)        plt.tight_layout()    (METRICS_DIR / "../figures").mkdir(parents=True, exist_ok=True)    plt.savefig(METRICS_DIR / f"../figures/component_breakdown_{sample_query}.png", dpi=150, bbox_inches="tight")    plt.show()

In [ ]:
if breakdown_df is not None:
    # Display formatted table
    display_df = breakdown_df.copy()
    display_df.columns = ["Ranker", "Return Sim (%)", "Sector Sim (%)", "Liq. Improve (%)", "Total (%)"]
    display_df["Total (%)"] = (display_df["Return Sim (%)"] + 
                               display_df["Sector Sim (%)"] + 
                               display_df["Liq. Improve (%)"]).round(1)
    
    print("\n### Component Breakdown by Ranker\n")
    print(display_df.to_string(index=False))

---
## 4. Per-Query Component Drill-Down

Examine component breakdown for individual queries to see how each ranker performs on specific stocks.

In [ ]:
# Load detailed breakdown
breakdown_detail_path = METRICS_DIR / "utility_breakdown_analysis.csv"

if breakdown_detail_path.exists():
    breakdown_detail = pd.read_csv(breakdown_detail_path)
    
    # Show available queries
    queries = breakdown_detail["query"].unique()
    print(f"Available queries ({len(queries)} total):")
    print(", ".join(queries[:20]), "..." if len(queries) > 20 else "")
    
    # Pick a sample query for analysis
    sample_query = queries[0]
    print(f"\nSample query: {sample_query}")
else:
    print(f"Warning: {breakdown_detail_path} not found")
    breakdown_detail = None

In [ ]:
if breakdown_detail is not None:    # Filter for sample query    query_data = breakdown_detail[breakdown_detail["query"] == sample_query]        # Create grouped bar chart    fig, ax = plt.subplots(figsize=(12, 6))        x = np.arange(len(query_data))    width = 0.25        ax.bar(x - width, query_data["pct_return_sim"], width, label="Return Similarity", color="#2E86AB")    ax.bar(x, query_data["pct_sector_sim"], width, label="Sector Similarity", color="#A23B72")    ax.bar(x + width, query_data["pct_liq_improve"], width, label="Liquidity Improvement", color="#06A77D")        ax.set_xlabel("Ranker", fontweight="bold")    ax.set_ylabel("Percentage (%)", fontweight="bold")    ax.set_title(f"Component Breakdown for Query: {sample_query}", fontsize=14, fontweight="bold")    ax.set_xticks(x)    ax.set_xticklabels(query_data["ranker"].map(RANKER_LABELS), rotation=45, ha="right")    ax.legend()    ax.set_ylim(0, 105)        plt.tight_layout()    (METRICS_DIR / "../figures").mkdir(parents=True, exist_ok=True)    plt.savefig(METRICS_DIR / f"../figures/component_breakdown_{sample_query}.png", dpi=150, bbox_inches="tight")    plt.show()        # Show raw data table    print(f"\n### Raw Data for Query {sample_query}\n")    display_data = query_data[["ranker", "pct_return_sim", "pct_sector_sim", "pct_liq_improve"]].copy()    display_data.columns = ["Ranker", "Return Sim (%)", "Sector Sim (%)", "Liq. Improve (%)"]    display_data["Ranker"] = display_data["Ranker"].map(RANKER_LABELS)    print(display_data.to_string(index=False))

---
## 5. Overall Metrics — All Rankers

In [ ]:
rankers = [r for r in RANKER_LABELS if r in overall_df.columns]
metrics = overall_df["metric_name"].tolist()

n_metrics = len(metrics)
n_rankers = len(rankers)
x = np.arange(n_metrics)
total_width = 0.7
bar_w = total_width / n_rankers

fig, ax = plt.subplots(figsize=(12, 5))

for i, ranker in enumerate(rankers):
    offset = (i - n_rankers / 2 + 0.5) * bar_w
    vals = overall_df[ranker].values
    bars = ax.bar(
        x + offset, vals, bar_w,
        label=RANKER_LABELS[ranker],
        color=RANKER_COLORS[i],
    )
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                max(v, 0) + 0.008,
                f"{v:.3f}",
                ha="center", va="bottom", fontsize=7.5, rotation=90
            )

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylabel("Score")
ax.set_title("Overall Retrieval Metrics — Ranker Comparison", fontweight="bold")
ax.axhline(0, color="black", linewidth=0.6)
ax.legend(loc="upper left", fontsize=9, ncol=2)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
plt.tight_layout()
plt.show()

In [ ]:
# --- Heatmap view for quick comparison ---

heatmap_data = overall_df.set_index("metric_name")[rankers].rename(columns=RANKER_LABELS)

fig, ax = plt.subplots(figsize=(10, 3))
sns.heatmap(
    heatmap_data,
    annot=True, fmt=".3f",
    cmap="RdYlGn",
    vmin=0, vmax=0.6,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Retrieval Metrics Heatmap", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

---
## 6. Metric Deep-Dives

In [ ]:
def metric_row(df, metric):
    row = df[df["metric_name"] == metric]
    if row.empty:
        return pd.Series(dtype=float)
    return row.iloc[0][rankers]

recall_vals  = metric_row(overall_df, "Recall@10")
spearman_vals = metric_row(overall_df, "Spearman")
ndcg_vals    = metric_row(overall_df, "nDCG@10")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, vals, title, ylabel in [
    (axes[0], recall_vals,   "Recall@10",  "Recall"),
    (axes[1], spearman_vals, "Spearman ρ", "Correlation"),
    (axes[2], ndcg_vals,     "nDCG@10",    "nDCG"),
]:
    labels = [RANKER_LABELS[r] for r in rankers]
    colors = RANKER_COLORS[:len(rankers)]
    bars = ax.bar(labels, vals.values, color=colors)
    ax.axhline(0, color="black", linewidth=0.6)
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
    for bar, v in zip(bars, vals.values):
        if not np.isnan(v):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                max(v, 0) + 0.005,
                f"{v:.3f}",
                ha="center", va="bottom", fontsize=8
            )

plt.suptitle("Per-Metric Ranker Comparison", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 7. Per-Reference Signal Comparison (Detailed)

How each ranker performs against each ground-truth reference independently.

In [ ]:
# Grouped bar: for each metric, compare rankers across the 3 reference signals

METRICS = ["Recall@10", "nDCG@10", "Spearman"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric in zip(axes, METRICS):
    vals = {}
    for ref_key, ref_df in ref_dfs.items():
        row = ref_df[ref_df["metric_name"] == metric]
        if row.empty:
            continue
        vals[ref_key] = [float(row[r].values[0]) if r in row.columns else 0.0 for r in rankers]

    x = np.arange(len(rankers))
    bar_w = 0.65 / len(vals)
    ref_colors = ["#4E9AF1", "#FF7043", "#66BB6A"]

    for j, (ref_key, v) in enumerate(vals.items()):
        offset = (j - len(vals) / 2 + 0.5) * bar_w
        ax.bar(x + offset, v, bar_w, label=REF_LABELS[ref_key], color=ref_colors[j], alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels([RANKER_LABELS[r] for r in rankers], rotation=35, ha="right", fontsize=8)
    ax.set_title(metric, fontweight="bold")
    ax.axhline(0, color="black", linewidth=0.5)
    ax.legend(fontsize=8)

plt.suptitle("Metrics per Reference Signal (bars = references, positions = rankers)", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: Reference × Ranker for each metric (3 heatmaps)

fig, axes = plt.subplots(1, 3, figsize=(20, 4))

for ax, metric in zip(axes, METRICS):
    rows = {}
    for ref_key, ref_df in ref_dfs.items():
        row = ref_df[ref_df["metric_name"] == metric]
        if not row.empty:
            rows[REF_LABELS[ref_key]] = {RANKER_LABELS[r]: float(row[r].values[0]) for r in rankers if r in row.columns}

    pivot = pd.DataFrame(rows).T  # shape: ref × ranker

    sns.heatmap(
        pivot,
        annot=True, fmt=".3f",
        cmap="YlOrRd",
        linewidths=0.4,
        ax=ax,
        vmin=0,
    )
    ax.set_title(f"{metric} — Reference × Ranker", fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=40, labelsize=8)

plt.tight_layout()
plt.show()

---
## 8. Utility Breakdown — Return, Sector, Liquidity

In [ ]:
# Stacked bar: % return-similar, % same-sector, % liquidity-improved — per ranker

bk = util_bk.copy()
bk = bk[bk["ranker"].isin(rankers)].copy()
bk["ranker_label"] = bk["ranker"].map(RANKER_LABELS)
bk = bk.set_index("ranker_label").reindex([RANKER_LABELS[r] for r in rankers if r in bk["ranker"].values])

x = np.arange(len(bk))
bar_w = 0.55

fig, ax = plt.subplots(figsize=(12, 5))
cols = [("pct_return_sim", "#2E86AB", "% Return Similar"),
        ("pct_sector_sim", "#A23B72", "% Same Sector"),
        ("pct_liq_improve", "#06A77D", "% Liquidity Improved")]

bottom = np.zeros(len(bk))
for col, color, label in cols:
    if col not in bk.columns:
        continue
    vals = bk[col].values
    ax.bar(x, vals, bar_w, bottom=bottom, label=label, color=color, alpha=0.88)
    for xi, (v, b) in enumerate(zip(vals, bottom)):
        if v > 3:
            ax.text(xi, b + v / 2, f"{v:.0f}%", ha="center", va="center", fontsize=8, color="white", fontweight="bold")
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels(bk.index.tolist(), rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Percentage of top-10 results (%)")
ax.set_title("Top-10 Utility Breakdown by Ranker", fontweight="bold")
ax.legend(fontsize=9, loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Utility breakdown table
print("Utility Breakdown — Averaged across queries:")
display_bk = util_bk.copy()
display_bk["ranker"] = display_bk["ranker"].map(lambda r: RANKER_LABELS.get(r, r))
display(display_bk.set_index("ranker"))

# nDCG@10 heatmap: ranker × reference
ndcg_rows = {}
for ref_key, ref_df in ref_dfs.items():
    row = ref_df[ref_df["metric_name"] == "nDCG@10"]
    if not row.empty:
        ndcg_rows[REF_LABELS[ref_key]] = {RANKER_LABELS[r]: float(row[r].values[0]) for r in rankers if r in row.columns}

pivot_ndcg = pd.DataFrame(ndcg_rows).T

fig, ax = plt.subplots(figsize=(12, 3))
sns.heatmap(
    pivot_ndcg,
    annot=True, fmt=".3f",
    cmap="YlGnBu",
    linewidths=0.4,
    ax=ax,
    vmin=0, vmax=1,
)
ax.set_title("nDCG@10 by Reference Signal × Ranker", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", rotation=35, labelsize=8)
plt.tight_layout()
plt.show()

---
## 9. Radar / Spider Chart — Overall Ranker Profile

Compares all rankers across all three metrics at once.

In [ ]:
metric_names = overall_df["metric_name"].tolist()
N = len(metric_names)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # close the polygon

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for i, ranker in enumerate(rankers):
    vals_raw = overall_df[ranker].clip(lower=0).tolist()
    vals_raw += vals_raw[:1]
    ax.plot(angles, vals_raw, "o-", linewidth=2,
            color=RANKER_COLORS[i], label=RANKER_LABELS[ranker])
    ax.fill(angles, vals_raw, alpha=0.07, color=RANKER_COLORS[i])

ax.set_thetagrids(np.degrees(angles[:-1]), metric_names, fontsize=12)
ax.set_title("Ranker Profile Radar", fontweight="bold", pad=18)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=9)
plt.tight_layout()
plt.show()

---
## 10. Re-run Retrieval Evaluation (optional)

Set paths below to regenerate metrics from raw features and a checkpoint.

In [ ]:
CHECKPOINT_PATH = None   # e.g. Path("checkpoints/last.ckpt")
FEATURES_PATH   = None   # e.g. Path("data/processed/all_features.parquet")
OUTPUT_DIR      = PROJECT_ROOT / "results" / "retrieval_v2"

if CHECKPOINT_PATH is not None and FEATURES_PATH is not None:
    import subprocess, sys
    result = subprocess.run(
        [
            sys.executable, "-m",
            "scripts.evaluation.run_retrieval_metrics",
            "--features", str(FEATURES_PATH),
            "--checkpoint", str(CHECKPOINT_PATH),
            "--output-dir", str(OUTPUT_DIR),
        ],
        capture_output=True, text=True, cwd=str(PROJECT_ROOT)
    )
    print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])
    else:
        # Reload freshly generated CSVs
        overall_df = pd.read_csv(OUTPUT_DIR / "metrics" / "retrieval_metrics_overall.csv")
        print("\nReloaded fresh overall metrics:")
        print(overall_df)
else:
    print("Skipped — set CHECKPOINT_PATH and FEATURES_PATH to re-run.")

---
## 11. Summary & Interpretation (v2 — 6 rankers × 3 references)

Numbers below are **averaged across all 3 reference signals** (Similarity, LiquidityUplift, Utility).

| Ranker | Recall@10 | Spearman ρ | nDCG@10 | Verdict |
|--------|-----------|------------|---------|---------|
| Embedding | ~0.0053 | ~0.026 | ~0.436 | Reasonable nDCG baseline; low binary recall |
| Pearson Corr. | ~0.0063 | ~0.260 | ~0.547 | Better Spearman; good linear co-movement signal |
| Spearman Corr. | ~0.0061 | ~0.273 | ~0.544 | Similar to Pearson — monotonic variant, marginally better Spearman |
| Emb. + Rerank | ~0.0096 | ~0.645 | ~0.661 | Reranking by liquidity proximity lifts Spearman and Recall substantially |
| Pearson + Rerank | ~0.0096 | ~0.648 | ~0.789 | Best overall — strongest nDCG and high Spearman |
| Spearman + Rerank | ~0.0101 | ~0.655 | ~0.806 | Highest nDCG and Spearman; best Recall@10 |

**Key takeaways:**
- Raw embedding Spearman (~0.026) is near-zero, meaning the model's cosine rankings don't yet correlate well with any reference signal on average.
- Adding liquidity re-ranking dramatically lifts all three metrics (Spearman: 0.026 → 0.645+, nDCG: 0.44 → 0.79+).
- Correlation-based rankers (Pearson/Spearman + Rerank) outperform embedding + rerank, suggesting the temporal encoder is not yet the dominant signal.
- The Utility reference drives the highest nDCG scores for rerankers (~0.78–0.91), reflecting that re-ranking inherently optimizes the utility criteria.
- **LiquidityUplift Spearman for rerankers = ~1.0**: this is expected — the reranker sorts by liquidity proximity, which exactly matches the LiquidityUplift ordering.

**Next step:** Block 5 (Crisis Spearman) — test whether embedding-based rankings stay stable across regime shifts where return correlation breaks down.